# B8 — Model Card : documenter le modèle de détection d'anomalies

**Prérequis** : TP4 (checkpoint validé `ae_v3_best.keras`), TP6 (mesure CodeCarbon),
TP7 (`config.py` + `scripts/run_vision_pipeline.py`).

> *Product Owner : le modèle est validé, mais personne d'autre que toi ne sait à quoi il
> sert, ce qu'il vaut, ni ce qu'il ne faut pas lui faire faire.*

Plutôt qu'inventer un format, on remplit le **template standard Hugging Face**
(`huggingface_hub.ModelCard.from_template`) — celui que tout le monde reconnaît, avec
usage, données, métriques, limites et impact déjà structurés.

| § | Contenu |
|---|---|
| §1 | Rassembler les éléments — modèle, données, métriques, CodeCarbon |
| §2 | Le template Hugging Face officiel |
| §3 | Usage (Direct / Downstream / Out-of-Scope), biais, risques, limites |
| §4 | Évaluation, impact environnemental, version, contact — génération finale |


## §0 — Setup

In [1]:
import os, sys, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from pathlib import Path
import yaml
import tensorflow as tf
from tensorflow import keras

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "src"))

import config
from indusense.vision.dataset import load_train_val, load_test
from indusense.vision.model import mse_ssim_loss, compression_ratio
from indusense.vision.anomaly import reconstruction_errors, calibrate_threshold, evaluate

from huggingface_hub import ModelCard, ModelCardData
from huggingface_hub.repocard_data import EvalResult

print(f"TF {tf.__version__}")
print(f"Template HF officiel : {ModelCard.default_template_path}")


TF 2.21.0
Template HF officiel : C:\Users\Aelion\py-init\.venv\Lib\site-packages\huggingface_hub\templates\modelcard_template.md


## §1 — Rassembler les éléments

Plutôt que copier-coller des chiffres d'un notebook précédent, on **recharge le modèle
validé et on recalcule ses métriques à l'identique du TP4** (mêmes fonctions
`indusense.vision`, même seed, même seuil) — la model card documente ce qui est
*vérifiable maintenant*, pas ce qui a été noté un jour.

In [2]:
loss_fn_v3 = mse_ssim_loss(alpha=0.8)
model_v3 = keras.models.load_model(
    config.CHECKPOINTS_DIR / "ae_v3_best.keras",
    custom_objects={loss_fn_v3.__name__: loss_fn_v3},  # "mse80_ssim19" — int((1-0.8)*100) tronque via l'arrondi flottant
)
print(f"Modèle chargé : {model_v3.count_params():,} paramètres — ratio compression {compression_ratio(model_v3):.1f}x")

X_train, X_val = load_train_val(config.BOTTLE_ROOT, val_ratio=config.VAL_RATIO, seed=config.RANDOM_SEED)
X_test, y_test, test_classes = load_test(config.BOTTLE_ROOT)

errors_val  = reconstruction_errors(model_v3, X_val)
errors_test = reconstruction_errors(model_v3, X_test)
threshold   = calibrate_threshold(errors_val, method="percentile", percentile=99)
metrics     = evaluate(y_test, errors_test, threshold)

print(json.dumps(metrics, indent=2))


Modèle chargé : 334,019 paramètres — ratio compression 12.0x


{
  "auroc": 0.6182539682539683,
  "threshold": 0.0008128880872391164,
  "tp": 14,
  "tn": 19,
  "fp": 1,
  "fn": 49,
  "recall": 0.2222222222222222,
  "precision": 0.9333333333333333,
  "specificity": 0.95
}


In [3]:
dataset_card = yaml.safe_load((config.ROOT_DIR / "dataset_card.yaml").read_text(encoding="utf-8"))
print(json.dumps(dataset_card, indent=2, ensure_ascii=False))


{
  "version_id": "bottle-v1.0",
  "source": {
    "name": "MVTec Anomaly Detection Dataset",
    "url": "https://www.mvtec.com/company/research/datasets/mvtec-ad",
    "category": "bottle",
    "license": "CC BY-NC-SA 4.0"
  },
  "acquisition": {
    "date_downloaded": null,
    "sensor": null,
    "resolution_original": [
      900,
      900
    ]
  },
  "preprocessing": {
    "target_size": [
      256,
      256
    ],
    "resize_strategy": "padding",
    "interpolation_images": "LANCZOS",
    "interpolation_masks": "NEAREST"
  },
  "split": {
    "seed": 42,
    "val_ratio": 0.2,
    "counts": {
      "train_good": 167,
      "val_good": 42,
      "test_good": 20,
      "test_broken_large": 20,
      "test_broken_small": 22,
      "test_contamination": 21,
      "masks": 63
    }
  },
  "normalization": {
    "mean_rgb": null,
    "std_rgb": null
  }
}


**Mesure CodeCarbon.** L'entraînement exact qui a produit `ae_v3_best.keras` (TP4, 50
epochs, données augmentées) précède l'instrumentation CodeCarbon — il n'a **pas** été
mesuré. Le TP6 a mesuré un run représentatif (même architecture, 10 epochs, sans
augmentation) : **0.0515 gCO2eq / 0.918 Wh** (mix France, 56 gCO2/kWh). On documente
cette mesure telle quelle, avec sa provenance exacte — pas d'extrapolation présentée
comme une mesure directe.

In [4]:
CODECARBON_MEASURE = {
    "source": "TP6.ipynb §1 — run représentatif, PAS le run exact du checkpoint ae_v3_best.keras",
    "hardware": "Intel Core i7-12700H (CPU) + NVIDIA RTX 3050 Ti Laptop (non utilisé par TF, TF>=2.11 sans support GPU natif Windows)",
    "emissions_gco2eq": 0.0515,
    "energy_wh": 0.918,
    "duration_s": 94.8,
    "epochs_measured": 10,
    "country": "France (FRA)",
    "grid_intensity_gco2_kwh": 56,
}
print(json.dumps(CODECARBON_MEASURE, indent=2, ensure_ascii=False))


{
  "source": "TP6.ipynb §1 — run représentatif, PAS le run exact du checkpoint ae_v3_best.keras",
  "hardware": "Intel Core i7-12700H (CPU) + NVIDIA RTX 3050 Ti Laptop (non utilisé par TF, TF>=2.11 sans support GPU natif Windows)",
  "emissions_gco2eq": 0.0515,
  "energy_wh": 0.918,
  "duration_s": 94.8,
  "epochs_measured": 10,
  "country": "France (FRA)",
  "grid_intensity_gco2_kwh": 56
}


## §2 — Le template Hugging Face officiel

`huggingface_hub` embarque le squelette officiel — on ne le réécrit pas, on le remplit
via `ModelCard.from_template(card_data, **template_kwargs)`.

In [5]:
template_head = Path(ModelCard.default_template_path).read_text(encoding="utf-8")
print(template_head[:900])
print("...")
print(f"\n[{len(template_head.splitlines())} lignes au total, {template_head.count('{{')} champs {{{{ }}}} à remplir]")


---
# For reference on model card metadata, see the spec: https://github.com/huggingface/hub-docs/blob/main/modelcard.md?plain=1
# Doc / guide: https://huggingface.co/docs/hub/model-cards
{{ card_data }}
---

# Model Card for {{ model_id | default("Model ID", true) }}

<!-- Provide a quick summary of what the model is/does. -->

{{ model_summary | default("", true) }}

## Model Details

### Model Description

<!-- Provide a longer summary of what this model is. -->

{{ model_description | default("", true) }}

- **Developed by:** {{ developers | default("[More Information Needed]", true)}}
- **Funded by [optional]:** {{ funded_by | default("[More Information Needed]", true)}}
- **Shared by [optional]:** {{ shared_by | default("[More Information Needed]", true)}}
- **Model type:** {{ model_type | default("[More Information Needed]", true)}}
- **Language(s) (NLP):** {{ language | default("
...

[200 lignes au total, 45 champs {{ }} à remplir]


## §3 — Usage, biais, risques, limites

*Point de réflexion (Étape 3 du md)* : un lecteur non technique comprend-il **quand faire
confiance au modèle et quand se méfier** ? On documente les trois usages (Direct /
Downstream / Out-of-Scope) et on est honnête sur les limites — y compris le fait qu'une
alternative interne (PatchCore-lite, TP4 §3) obtient un AUROC bien supérieur.

In [6]:
card_data = ModelCardData(
    model_name="indusense-ae-bottle-v3",
    license="cc-by-nc-sa-4.0",
    library_name="keras",
    tags=["computer-vision", "anomaly-detection", "autoencoder", "manufacturing", "mvtec-ad"],
    datasets=["mvtec-ad"],
    eval_results=[
        EvalResult(
            task_type="anomaly-detection",
            dataset_type="mvtec-ad-bottle",
            dataset_name="MVTec AD — bottle",
            metric_type="auroc",
            metric_value=round(metrics["auroc"], 3),
            metric_name="AUROC (image-level)",
        ),
    ],
)

template_kwargs = dict(
    model_id="indusense-ae-bottle-v3",
    model_summary=(
        "Auto-encodeur convolutionnel pour la détection d'anomalies visuelles sur la "
        "catégorie *bottle* de MVTec AD. Score d'anomalie = erreur de reconstruction "
        "(MSE + SSIM) ; un score au-delà d'un seuil calibré signale un défaut candidat."
    ),
    model_description=(
        "Le modèle apprend à reconstruire des images de bouteilles **saines uniquement**. "
        "Un goulot d'étranglement resserré (ratio de compression "
        f"{compression_ratio(model_v3):.0f}x) empêche une reconstruction fidèle des défauts "
        "absents de l'entraînement : leur erreur de reconstruction est donc statistiquement "
        "plus élevée que celle des pièces saines. Voir TP3/TP4 pour la démarche complète."
    ),
    developers="Guillaume Saïdani",
    model_type="Auto-encodeur convolutionnel (Conv2D / Conv2DTranspose), perte MSE+SSIM",
    language="n/a (vision par ordinateur, pas de NLP)",
    license="CC BY-NC-SA 4.0 (héritée du dataset MVTec AD — usage non commercial)",
    base_model="Aucun — entraîné from scratch (pas de fine-tuning)",
    repo="DL/ (ce dépôt) — checkpoints/ae_v3_best.keras",
    direct_use=(
        "Scorer une image de bouteille (256x256, RGB, [0,1]) et comparer son erreur de "
        "reconstruction MSE+SSIM au seuil calibré "
        f"({threshold:.5f}, percentile 99 sur validation saine) pour obtenir un signal "
        "*candidat défaut / probablement sain*, à l'usage exclusif d'un opérateur humain "
        "en contrôle qualité."
    ),
    downstream_use=(
        "Intégration dans un tableau de bord de contrôle qualité comme signal d'aide à la "
        "décision (score + heatmap d'erreur, cf. TP3/TP4), en complément d'une inspection "
        "humaine — ou combiné à PatchCore-lite (TP4 §3, AUROC 0.859) pour un score ensembliste."
    ),
    out_of_scope_use=(
        "- Décision automatique de rejet/acceptation sans supervision humaine.\n"
        "- Toute catégorie MVTec AD autre que *bottle*, ou tout produit hors MVTec AD, "
        "sans ré-entraînement et recalibration complète du seuil.\n"
        "- Détection de types de défauts absents du jeu de test "
        "(seuls broken_large, broken_small, contamination sont couverts).\n"
        "- Usage réglementaire ou de certification sécurité — ce modèle n'a fait l'objet "
        "d'aucune validation de ce type."
    ),
    bias_risks_limitations=(
        f"- **Performance modeste** : AUROC = {metrics['auroc']:.3f} sur le test MVTec AD "
        f"bottle (rappel {metrics['recall']:.1%}, {metrics['fn']} défauts non détectés sur "
        f"{metrics['fn']+metrics['tp']}). Une alternative interne, PatchCore-lite "
        "(backbone ResNet50 pré-entraîné, TP4 §3), atteint un AUROC de 0.859 sur les mêmes "
        "données — ce modèle-ci reste documenté pour sa valeur pédagogique et parce qu'il "
        "est celui industrialisé par `scripts/run_vision_pipeline.py` (TP7), pas comme "
        "la meilleure option disponible.\n"
        "- **Risque d'identité résiduel** : un goulot trop large réapprend une quasi-copie "
        "de l'entrée (cf. TP3, AUROC 0.465 avant resserrement) ; le ratio actuel "
        f"({compression_ratio(model_v3):.0f}x) réduit ce risque sans l'éliminer totalement.\n"
        "- **Seuil sensible à la distribution** : calibré au 99e percentile des erreurs de "
        "validation saine — un changement d'éclairage, de fond ou de caméra en production "
        "peut invalider ce seuil sans que le modèle ne le signale.\n"
        "- **Un seul produit** : entraîné et évalué uniquement sur *bottle* — aucune garantie "
        "de généralisation à d'autres catégories ou lignes de production."
    ),
    bias_recommendations=(
        "Toujours faire valider les alertes par un opérateur humain. Recalibrer le seuil "
        "(`calibrate_threshold`) après tout changement d'éclairage/caméra/fond. Envisager "
        "PatchCore-lite (TP4 §3) si le rappel de ce modèle est insuffisant pour l'usage visé."
    ),
    get_started_code=(
        "```python\n"
        "from tensorflow import keras\n"
        "from indusense.vision.model import mse_ssim_loss\n"
        "from indusense.vision.anomaly import reconstruction_errors\n\n"
        "model = keras.models.load_model('checkpoints/ae_v3_best.keras',\n"
        "    custom_objects={'mse80_ssim19': mse_ssim_loss(alpha=0.8)})\n"
        "errors = reconstruction_errors(model, images)  # images: (N,256,256,3) float32 [0,1]\n"
        f"is_anomaly = errors >= {threshold:.5f}\n"
        "```"
    ),
)
print("card_data et template_kwargs (§3) prêts —", len(template_kwargs), "champs renseignés")


card_data et template_kwargs (§3) prêts — 15 champs renseignés


## §4 — Évaluation, impact environnemental, version, contact

In [7]:
n_train, n_val = len(X_train), len(X_val)
n_test = len(X_test)

template_kwargs.update(dict(
    training_data=(
        f"MVTec AD, catégorie *bottle* — {n_train} images saines (entraînement), "
        f"{n_val} images saines (validation), split fixe (seed={config.RANDOM_SEED}, "
        f"val_ratio={config.VAL_RATIO}). Licence CC BY-NC-SA 4.0. "
        "Voir `dataset_card.yaml` pour la traçabilité complète (source, prétraitement, split)."
    ),
    preprocessing=(
        "Redimensionnement 256x256 par padding centré (letterbox, ratio préservé), "
        "interpolation LANCZOS, normalisation [0,1] (division par 255)."
    ),
    training_regime="fp32, Adam (lr=1e-3), EarlyStopping (patience=10) + ReduceLROnPlateau, augmentation Albumentations (flip, rotation, luminosité, teinte, bruit)",
    speeds_sizes_times=f"Checkpoint : {(config.CHECKPOINTS_DIR / 'ae_v3_best.keras').stat().st_size / 1e6:.1f} Mo — {model_v3.count_params():,} paramètres",
    testing_data=(
        f"MVTec AD, catégorie *bottle*, split test complet — {n_test} images "
        f"({metrics['tn']+metrics['fp']} saines, {metrics['tp']+metrics['fn']} défectueuses : "
        "broken_large, broken_small, contamination)."
    ),
    testing_factors="Aucune stratification par sous-population — évaluation globale toutes classes de défaut confondues.",
    testing_metrics="AUROC (image-level), rappel, précision, spécificité — seuil calibré au 99e percentile des erreurs de reconstruction sur validation saine.",
    results=(
        f"AUROC = {metrics['auroc']:.3f} · Seuil = {threshold:.5f} · "
        f"TP={metrics['tp']} TN={metrics['tn']} FP={metrics['fp']} FN={metrics['fn']} · "
        f"Rappel = {metrics['recall']:.1%} · Précision = {metrics['precision']:.1%} · "
        f"Spécificité = {metrics['specificity']:.1%}"
    ),
    results_summary=(
        f"Précision {metrics['precision']:.0%}, rappel {metrics['recall']:.0%} : le modèle "
        "ne se trompe quasiment jamais quand il signale un défaut, mais en manque une "
        "majorité. Adapté à un usage de pré-filtrage assisté, pas de détection exhaustive autonome."
    ),
    model_examination="Non réalisé pour ce modèle — voir TP5 (SHAP GradientExplainer) pour une analyse d'attribution pixel sur cette même architecture.",
    hardware_type=CODECARBON_MEASURE["hardware"],
    hours_used=f"{CODECARBON_MEASURE['duration_s']/3600:.4f} h ({CODECARBON_MEASURE['epochs_measured']} epochs, run représentatif — voir note §1)",
    cloud_provider="Aucun — poste de travail local",
    cloud_region=f"{CODECARBON_MEASURE['country']}",
    co2_emitted=f"{CODECARBON_MEASURE['emissions_gco2eq']:.4f} gCO2eq ({CODECARBON_MEASURE['energy_wh']:.3f} Wh, mix {CODECARBON_MEASURE['grid_intensity_gco2_kwh']} gCO2/kWh) — mesure TP6, run représentatif non-identique au run exact du checkpoint",
    model_specs=f"Auto-encodeur : encodeur/décodeur symétrique filters=(32,64,128,64), goulot 16x16x64, ratio compression {compression_ratio(model_v3):.0f}x, perte 0.8*MSE + 0.2*(1-SSIM)",
    compute_infrastructure="Poste de travail local (pas de cluster) — reproductible via `scripts/run_vision_pipeline.py` (TP7)",
    hardware_requirements="CPU suffisant (pas de dépendance GPU stricte) — TensorFlow >=2.11 sans support GPU natif Windows dans cet environnement",
    software="TensorFlow 2.21, Python 3.13, indusense.vision (ce dépôt)",
    model_card_authors="Guillaume Saïdani",
    model_card_contact="guillaume.saidani@ext.aelion.fr",
))

card = ModelCard.from_template(card_data, **template_kwargs)
card.save(config.REPORTS_DIR / "model_card.md")
print(f"Model card sauvegardée : {config.REPORTS_DIR / 'model_card.md'}")
print(f"Longueur : {len(str(card))} caractères")


Model card sauvegardée : C:\Users\Aelion\py-init\DL\reports\model_card.md
Longueur : 10135 caractères


In [8]:
# Validation : le front-matter YAML doit parser et les eval_results doivent être exploitables
card.validate()
print("Validation OK — front-matter YAML conforme au schéma Hugging Face")
print()
print(str(card)[:1600])
print("...")


Validation OK — front-matter YAML conforme au schéma Hugging Face

---
datasets:
- mvtec-ad
library_name: keras
license: cc-by-nc-sa-4.0
tags:
- computer-vision
- anomaly-detection
- autoencoder
- manufacturing
- mvtec-ad
model-index:
- name: indusense-ae-bottle-v3
  results:
  - task:
      type: anomaly-detection
    dataset:
      name: MVTec AD — bottle
      type: mvtec-ad-bottle
    metrics:
    - type: auroc
      value: 0.618
      name: AUROC (image-level)
---

# Model Card for indusense-ae-bottle-v3

<!-- Provide a quick summary of what the model is/does. -->

Auto-encodeur convolutionnel pour la détection d'anomalies visuelles sur la catégorie *bottle* de MVTec AD. Score d'anomalie = erreur de reconstruction (MSE + SSIM) ; un score au-delà d'un seuil calibré signale un défaut candidat.

## Model Details

### Model Description

<!-- Provide a longer summary of what this model is. -->

Le modèle apprend à reconstruire des images de bouteilles **saines uniquement**. Un goulot d

## Synthèse

**Livrables (repris du md) :**
1. Model card au format Hugging Face — `reports/model_card.md`, sections obligatoires renseignées.
2. Trois usages explicites — Direct (§3), Downstream (§3), Out-of-Scope (§3).
3. Sections **Bias, Risks, and Limitations** et **Evaluation** complétées avec des métriques
   **recalculées en direct** (pas copiées) et une mesure carbone **tracée à sa source exacte**
   (TP6, run représentatif — pas le run réel du checkpoint).

**Honnêteté délibérée** : la card documente un modèle à AUROC 0.618 tout en indiquant
qu'une alternative interne (PatchCore-lite, AUROC 0.859) est disponible et surperforme —
une model card ne sert pas à vendre le modèle documenté, mais à donner au lecteur de quoi
décider s'il convient à son usage.